# 🤖 MetaWorld ACT Training Pipeline - COMPLETE WORKING VERSION

Based on verified working code. Generates expert demonstrations for ACT training.

## ✅ Prerequisites
- Kaggle GPU: T4 x2 (recommended)
- Secrets: `HF_TOKEN`, `WANDB_API_KEY`

## 0️⃣ System Dependencies

In [1]:
# ==========================================
# SYSTEM DEPENDENCIES
# ==========================================

!apt-get update -qq
!apt-get install -y -qq libgl1-mesa-dev libgl1-mesa-glx libglew-dev \
                         libosmesa6-dev software-properties-common patchelf

print("✅ System dependencies installed")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libglx-dev:amd64.
(Reading database ... 129073 files and directories currently installed.)
Preparing to unpack .../00-libglx-dev_1.4.0-1_amd64.deb ...
Unpacking libglx-dev:amd64 (1.4.0-1) ...
Selecting previously unselected package libgl-dev:amd64.
Preparing to unpack .../01-libgl-dev_1.4.0-1_amd64.deb ...
Unpacking libgl-dev:amd64 (1.4.0-1) ...
Selecting previously unselected package libegl-dev:amd64.
Preparing to unpack .../02-libegl-dev_1.4.0-1_amd64.deb ...
Unpacking libegl-dev:amd64 (1.4.0-1) ...
Selecting previously unselected package libgles1:amd64.
Preparing to unpack .../03-libgles1_1.4.0-1_amd64.deb ...
Unpacking libgles1:amd64 (1.4.0-1) ...
Selecting previously unselected package libgles-dev:amd64.
Preparing to unpack .../04-libgles-dev_1.4.0-1_amd64

## 1️⃣ Environment Setup

In [2]:
# ==========================================
# ENVIRONMENT SETUP
# ==========================================

import os
import sys
from pathlib import Path

# CRITICAL: Set environment variables BEFORE any imports
os.environ['MUJOCO_GL'] = 'egl'
os.environ['LEROBOT_VIDEO_BACKEND'] = 'pyav'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['SVT_LOG'] = '0'
os.environ['FFREPORT'] = 'level=quiet'

# Get Kaggle secrets
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
    os.environ['WANDB_API_KEY'] = secrets.get_secret('WANDB_API_KEY')
    print("✅ Secrets loaded from Kaggle")
except:
    print("⚠️  Running outside Kaggle - set secrets manually")

# Login to W&B
import wandb
wandb.login(key=os.environ.get('WANDB_API_KEY', ''))

print("✅ Environment configured")

✅ Secrets loaded from Kaggle


/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

✅ Environment configured


## 2️⃣ Install Dependencies

In [3]:
# ==========================================
# CLONE & INSTALL LEROBOT
# ==========================================

!git clone https://github.com/huggingface/lerobot.git /kaggle/working/lerobot
%cd /kaggle/working/lerobot
!pip install -e . -q
!pip install metaworld wandb opencv-python imageio imageio-ffmpeg av -q

print("\n✅ All dependencies installed")

Cloning into '/kaggle/working/lerobot'...
remote: Enumerating objects: 45930, done.
remote: Counting objects: 100% (84/84), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 45930 (delta 35), reused 41 (delta 26), pack-reused 45846 (from 2)
Receiving objects: 100% (45930/45930), 232.14 MiB | 39.70 MiB/s, done.
Resolving deltas: 100% (29683/29683), done.
Filtering content: 100% (45/45), 69.03 MiB | 63.80 MiB/s, done.
/kaggle/working/lerobot
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
# ==========================================
# ADD SRC TO PYTHON PATH
# ==========================================

import sys
from pathlib import Path

LEROBOT_DIR = Path("/kaggle/working/lerobot")
SRC_DIR = LEROBOT_DIR / "src"

if SRC_DIR.exists():
    sys.path.insert(0, str(SRC_DIR))
    print(f"✅ Added to Python path: {SRC_DIR}")
else:
    print(f"❌ Warning: {SRC_DIR} not found!")

# Verify imports
try:
    from lerobot.envs.metaworld import MetaworldEnv, TASK_DESCRIPTIONS
    from lerobot.datasets.lerobot_dataset import LeRobotDataset
    print("✅ LeRobot imports successful!")
except ImportError as e:
    print(f"❌ Import error: {e}")
    raise

✅ Added to Python path: /kaggle/working/lerobot/src
✅ LeRobot imports successful!


## 3️⃣ Configuration

In [5]:
# ==========================================
# CONFIGURATION
# ==========================================

import numpy as np
from pathlib import Path

# Dataset configuration
TASK_NAME = "pick-place-v3"  # MetaWorld task
NUM_EPISODES = 50  # Number of successful episodes
MAX_EPISODE_STEPS = 500  # Max steps per episode
OBSERVATION_WIDTH = 480
OBSERVATION_HEIGHT = 480
FPS = 20

# Paths
WORK_DIR = Path("/kaggle/working")
ROOT_DIR = WORK_DIR / "data"
REPO_ID = f"lerobot/{TASK_NAME}"
DATASET_DIR = ROOT_DIR / REPO_ID

# HuggingFace
HF_USERNAME = "YOUR_HF_USERNAME"  # ⚠️ CHANGE THIS!
HF_DATASET_REPO = f"{HF_USERNAME}/metaworld-{TASK_NAME}-expert"

# Create directories
ROOT_DIR.mkdir(parents=True, exist_ok=True)

print("📋 Configuration:")
print(f"   Task: {TASK_NAME}")
print(f"   Episodes: {NUM_EPISODES}")
print(f"   Resolution: {OBSERVATION_WIDTH}x{OBSERVATION_HEIGHT}")
print(f"   Dataset: {DATASET_DIR}")

📋 Configuration:
   Task: pick-place-v3
   Episodes: 50
   Resolution: 480x480
   Dataset: /kaggle/working/data/lerobot/pick-place-v3


## 4️⃣ Custom Environment Wrapper

In [6]:
# ==========================================
# CUSTOM METAWORLD WRAPPER
# ==========================================

from lerobot.envs.metaworld import MetaworldEnv

class MetaworldEnvWithRawObs(MetaworldEnv):
    """
    Extended MetaworldEnv that captures raw internal state for expert policy.
    
    Key features:
    - Stores raw 39-dim observation for expert
    - Ensures randomization is enabled
    - Proper termination handling
    """

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._raw_obs = None
        
        # CRITICAL: Ensure randomization is enabled
        self._env._freeze_rand_vec = False
        print(f"   ✅ Randomization enabled: _freeze_rand_vec={self._env._freeze_rand_vec}")

    def reset(self, seed=None, **kwargs):
        """Reset with randomization. Captures raw state for expert policy."""
        observation, info = super().reset(seed=seed, **kwargs)
        self._raw_obs = self._env._get_obs()
        return observation, info

    def step(self, action):
        """Execute action without auto-reset on termination."""
        if action.ndim != 1:
            raise ValueError(f"Expected 1-D action, got shape {action.shape}")

        # Direct call to MetaWorld environment (bypass parent's auto-reset)
        raw_obs, reward, done, truncated, info = self._env.step(action)
        
        is_success = bool(info.get("success", 0))
        terminated = done or is_success
        
        info.update({
            "task": self.task,
            "done": done,
            "is_success": is_success,
        })
        
        observation = self._format_raw_obs(raw_obs)
        self._raw_obs = self._env._get_obs()
        
        return observation, reward, terminated, truncated, info

print("✅ Custom environment wrapper defined")

✅ Custom environment wrapper defined


## 5️⃣ Dataset Creation Functions

In [7]:
# ==========================================
# DATASET CREATION
# ==========================================

import shutil
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from lerobot.envs.metaworld import TASK_DESCRIPTIONS

def create_dataset(repo_id: str, task_name: str) -> LeRobotDataset:
    """Create a new LeRobotDataset with proper features."""
    dataset_path = ROOT_DIR / repo_id

    # Clean up existing dataset
    if dataset_path.exists():
        print(f"   ⚠️  Removing existing dataset at {dataset_path}")
        shutil.rmtree(dataset_path)

    # Define features
    features = {
        "observation.images.image": {
            "dtype": "video",
            "shape": (OBSERVATION_HEIGHT, OBSERVATION_WIDTH, 3),
            "names": ["height", "width", "channel"],
        },
        "observation.state": {
            "dtype": "float32",
            "shape": (4,),
            "names": ["x", "y", "z", "gripper"],
        },
        "action": {
            "dtype": "float32",
            "shape": (4,),
            "names": ["dx", "dy", "dz", "gripper"],
        },
        "next.reward": {
            "dtype": "float32",
            "shape": (1,),
            "names": ["reward"],
        },
        "next.success": {
            "dtype": "bool",
            "shape": (1,),
            "names": ["success"],
        },
    }

    dataset = LeRobotDataset.create(
        repo_id=repo_id,
        fps=FPS,
        root=dataset_path,
        features=features,
        robot_type="sawyer",
        use_videos=True,
    )
    
    print(f"   ✅ Dataset created at {dataset_path}")
    return dataset

print("✅ Dataset creation function ready")

✅ Dataset creation function ready


## 6️⃣ Expert Dataset Recording

In [8]:
# ==========================================
# EXPERT DATASET RECORDING
# ==========================================

def record_dataset(task_name: str, num_episodes: int):
    """
    Record expert demonstrations for a MetaWorld task.
    
    Key features:
    - Only saves SUCCESSFUL episodes
    - Random seed per episode for diversity
    - Proper expert observation handling
    """
    repo_id = f"lerobot/{task_name}"
    task_description = TASK_DESCRIPTIONS.get(task_name, f"perform {task_name}")
    
    print(f"\n{'='*70}")
    print(f"🎬 Recording Expert Dataset")
    print(f"{'='*70}")
    print(f"   Task: {task_name}")
    print(f"   Description: '{task_description}'")
    print(f"   Target episodes: {num_episodes}")
    print(f"   Max steps/episode: {MAX_EPISODE_STEPS}")
    print(f"{'='*70}\n")

    # Initialize environment
    print("🔧 Initializing environment...")
    env = MetaworldEnvWithRawObs(
        task=task_name,
        camera_name="corner2",
        obs_type="pixels_agent_pos",
        render_mode="rgb_array",
        observation_width=OBSERVATION_WIDTH,
        observation_height=OBSERVATION_HEIGHT,
    )

    # Get expert policy
    try:
        expert_policy = env.expert_policy
        print(f"   ✅ Expert policy loaded: {type(expert_policy).__name__}")
    except AttributeError:
        print("   ❌ Error: Could not load expert policy")
        return None

    # Create dataset
    print(f"\n📁 Creating dataset...")
    dataset = create_dataset(repo_id, task_name)

    # Recording statistics
    success_count = 0
    total_attempts = 0
    total_frames = 0

    print(f"\n🚀 Starting recording...\n")

    try:
        while success_count < num_episodes:
            total_attempts += 1
            
            # Unique seed per episode for diversity
            seed = np.random.randint(0, 1_000_000)
            
            print(f"🎬 Episode attempt {total_attempts} (seed={seed})...", end=" ")
            
            obs, info = env.reset(seed=seed)
            dataset.episode_buffer = dataset.create_episode_buffer()

            done = False
            step_count = 0
            episode_success = False

            while not done and step_count < MAX_EPISODE_STEPS:
                # Extract observations
                image = obs["pixels"]
                agent_pos = obs["agent_pos"]

                # Get expert action using RAW internal state
                action = expert_policy.get_action(env._raw_obs)
                action = np.clip(action, -1.0, 1.0).astype(np.float32)

                # Step environment
                next_obs, reward, terminated, truncated, step_info = env.step(action)
                done = terminated or truncated

                # Check success
                if step_info.get("is_success", False) or step_info.get("success", 0) > 0.5:
                    episode_success = True

                # Create frame
                frame = {
                    "observation.images.image": image,
                    "observation.state": agent_pos.astype(np.float32),
                    "action": action,
                    "next.reward": np.array([reward], dtype=np.float32),
                    "next.success": np.array([episode_success]),
                    "task": task_description,
                }
                dataset.add_frame(frame)

                obs = next_obs
                step_count += 1

            # CRITICAL: Quality filtering
            if episode_success:
                success_count += 1
                total_frames += step_count
                dataset.save_episode()
                print(f"✅ SUCCESS | Steps: {step_count:3d} | Progress: {success_count}/{num_episodes}")
            else:
                # Discard failed episode
                dataset.episode_buffer = None
                print(f"❌ FAILED  | Steps: {step_count:3d} | Retrying...")

    except KeyboardInterrupt:
        print("\n\n⚠️  Recording interrupted by user")
    
    finally:
        print(f"\n{'='*70}")
        print("💾 Finalizing dataset...")
        
        # CRITICAL: Call finalize() before closing environment
        dataset.finalize()
        env.close()

        # Summary statistics
        print(f"\n📊 Dataset Summary:")
        print(f"   ✅ Successful episodes: {success_count}")
        print(f"   🎯 Total attempts: {total_attempts}")
        if total_attempts > 0:
            success_rate = 100 * success_count / total_attempts
            print(f"   📈 Success rate: {success_rate:.1f}%")
        if success_count > 0:
            avg_frames = total_frames / success_count
            print(f"   📏 Avg frames/episode: {avg_frames:.1f}")
        print(f"   💾 Total frames: {total_frames:,}")
        print(f"   📁 Saved to: {ROOT_DIR / repo_id}")
        print(f"{'='*70}\n")
        print("✅ Done!")
    
    return ROOT_DIR / repo_id

print("✅ Recording function ready")

✅ Recording function ready


## 7️⃣ Run Dataset Generation

In [9]:
# ==========================================
# RUN DATASET GENERATION
# ==========================================

dataset_path = record_dataset(TASK_NAME, NUM_EPISODES)


🎬 Recording Expert Dataset
   Task: pick-place-v3
   Description: 'Pick and place a puck to a goal'
   Target episodes: 50
   Max steps/episode: 500

🔧 Initializing environment...
   ✅ Randomization enabled: _freeze_rand_vec=False
   ✅ Expert policy loaded: SawyerPickPlaceV3Policy

📁 Creating dataset...
   ✅ Dataset created at /kaggle/working/data/lerobot/pick-place-v3

🚀 Starting recording...

🎬 Episode attempt 1 (seed=106183)... 

/usr/local/lib/python3.12/dist-packages/metaworld/policies/policy.py:49: UserWarning: Constant(s) may be too high. Environments clip response to [-1, 1]
  warnings.warn(
/kaggle/working/lerobot/src/lerobot/datasets/compute_stats.py:154: RuntimeWarning: Converting input from bool to <class 'numpy.uint8'> for compatibility.
  hist, _ = np.histogram(batch[:, i], bins=self._bin_edges[i])


Map:   0%|          | 0/55 [00:00<?, ? examples/s]

✅ SUCCESS | Steps:  55 | Progress: 1/50
🎬 Episode attempt 2 (seed=445828)... 

Map:   0%|          | 0/56 [00:00<?, ? examples/s]

...

[mp4 @ 0x36060dc0] Starting second pass: moving the moov atom to the beginning of the file


✅ SUCCESS | Steps:  60 | Progress: 49/50
🎬 Episode attempt 50 (seed=570369)... 

Map:   0%|          | 0/54 [00:00<?, ? examples/s]

✅ SUCCESS | Steps:  54 | Progress: 50/50

💾 Finalizing dataset...

📊 Dataset Summary:
   ✅ Successful episodes: 50
   🎯 Total attempts: 50
   📈 Success rate: 100.0%
   📏 Avg frames/episode: 53.7
   💾 Total frames: 2,684
   📁 Saved to: /kaggle/working/data/lerobot/pick-place-v3

✅ Done!


[mp4 @ 0x39dc5bc0] Starting second pass: moving the moov atom to the beginning of the file


## 8️⃣ Verify Dataset

In [10]:
# ==========================================
# VERIFY DATASET (FINAL CORRECTED VERSION)
# ==========================================

from lerobot.datasets.lerobot_dataset import LeRobotDataset
import numpy as np

print(f"\n{'='*70}")
print("🔍 Verifying Dataset")
print(f"{'='*70}\n")

try:
    # Load dataset
    dataset = LeRobotDataset(
        repo_id=REPO_ID,
        root=DATASET_DIR,
        video_backend="pyav"
    )

    print(f"📊 Dataset Statistics:")
    print(f"   Episodes: {dataset.num_episodes}")
    print(f"   Total frames: {dataset.num_frames:,}")
    print(f"   FPS: {dataset.fps}")
    print(f"   Robot type: {dataset.meta.robot_type}")
    
    print(f"\n📋 Features:")
    for feature_name in dataset.features.keys():
        print(f"   - {feature_name}")
    
    # Test loading a sample
    if len(dataset) > 0:
        sample = dataset[0]
        print(f"\n🧪 Sample Frame:")
        print(f"   Task: '{sample['task']}'")
        print(f"   Image shape: {sample['observation.images.image'].shape}")
        print(f"   State shape: {sample['observation.state'].shape}")
        print(f"   Action shape: {sample['action'].shape}")
        
        action = sample['action'].numpy()
        print(f"   Action range: [{action.min():.3f}, {action.max():.3f}]")
    
    # Calculate episode lengths using episode_index
    if dataset.num_episodes > 0:
        print(f"\n📈 Episode Lengths:")
        
        # Get all episode indices from the dataset
        episode_lengths = []
        for ep_idx in range(dataset.num_episodes):
            # Count frames for this episode
            ep_length = 0
            for i in range(len(dataset)):
                frame = dataset[i]
                if frame['episode_index'].item() == ep_idx:
                    ep_length += 1
            episode_lengths.append(ep_length)
        
        # Show first 10
        for i in range(min(10, len(episode_lengths))):
            print(f"   Episode {i:2d}: {episode_lengths[i]:3d} frames")
        
        if len(episode_lengths) > 10:
            print(f"   ... and {len(episode_lengths) - 10} more episodes")
        
        # Statistics
        print(f"\n📊 Episode Length Statistics:")
        print(f"   Min: {min(episode_lengths)} frames")
        print(f"   Max: {max(episode_lengths)} frames")
        print(f"   Mean: {np.mean(episode_lengths):.1f} frames")
        print(f"   Median: {np.median(episode_lengths):.1f} frames")
        print(f"   Total: {sum(episode_lengths)} frames")

    print(f"\n{'='*70}")
    print("✅ Dataset verification PASSED!")
    print("✅ Ready for training!")
    print(f"{'='*70}\n")
    
except Exception as e:
    print(f"❌ Verification failed: {e}")
    import traceback
    traceback.print_exc()


🔍 Verifying Dataset

📊 Dataset Statistics:
   Episodes: 50
   Total frames: 2,684
   FPS: 20
   Robot type: sawyer

📋 Features:
   - observation.images.image
   - observation.state
   - action
   - next.reward
   - next.success
   - timestamp
   - frame_index
   - episode_index
   - index
   - task_index

🧪 Sample Frame:
   Task: 'Pick and place a puck to a goal'
   Image shape: torch.Size([3, 480, 480])
   State shape: torch.Size([4])
   Action shape: torch.Size([4])
   Action range: [-0.751, 0.811]

📈 Episode Lengths:


/usr/local/lib/python3.12/dist-packages/torchvision/io/_video_deprecation_warning.py:5: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.warn(


   Episode  0:  55 frames
   Episode  1:  56 frames
   Episode  2:  52 frames
   Episode  3:  56 frames
   Episode  4:  52 frames
   Episode  5:  61 frames
   Episode  6:  53 frames
   Episode  7:  54 frames
   Episode  8:  51 frames
   Episode  9:  50 frames
   ... and 40 more episodes

📊 Episode Length Statistics:
   Min: 45 frames
   Max: 61 frames
   Mean: 53.7 frames
   Median: 54.0 frames
   Total: 2684 frames

✅ Dataset verification PASSED!
✅ Ready for training!



## 9️⃣ Upload to HuggingFace (Optional)

In [11]:
# ==========================================
# FIX: SET YOUR ACTUAL HUGGINGFACE USERNAME
# ==========================================

# CHANGE THIS to your actual HuggingFace username!
HF_USERNAME = "aryannzzz"  # ⚠️ REPLACE WITH YOUR USERNAME

# Update repository name
HF_DATASET_REPO = f"{HF_USERNAME}/metaworld-{TASK_NAME}-expert"

print(f"✅ HuggingFace username set to: {HF_USERNAME}")
print(f"✅ Will upload to: {HF_DATASET_REPO}")

✅ HuggingFace username set to: aryannzzz
✅ Will upload to: aryannzzz/metaworld-pick-place-v3-expert


In [12]:
# ==========================================
# FIX: RELOAD DATASET WITH YOUR REPO ID
# ==========================================

from lerobot.datasets.lerobot_dataset import LeRobotDataset

print("🔄 Reloading dataset with correct repo_id...\n")

# Load the dataset from disk with YOUR repo_id
dataset = LeRobotDataset(
    repo_id=HF_DATASET_REPO,  # Use YOUR repo_id, not "lerobot/pick-place-v3"
    root=DATASET_DIR,
    video_backend="pyav"
)

# Now the dataset object has the correct repo_id internally
print(f"✅ Dataset loaded with repo_id: {dataset.repo_id}")
print(f"✅ Episodes: {dataset.num_episodes}")
print(f"✅ Frames: {dataset.num_frames:,}\n")

# Now push to hub
print(f"{'='*70}")
print("📤 Uploading Dataset to HuggingFace")
print(f"{'='*70}\n")
print(f"   Repository: {HF_DATASET_REPO}")

try:
    dataset.push_to_hub(
        token=os.environ['HF_TOKEN'],
        private=False,
    )
    
    print(f"\n{'='*70}")
    print("✅ Dataset uploaded successfully!")
    print(f"{'='*70}")
    print(f"\n🔗 View at: https://huggingface.co/datasets/{HF_DATASET_REPO}\n")
    
except Exception as e:
    print(f"\n⚠️  Upload failed: {e}")
    import traceback
    traceback.print_exc()

🔄 Reloading dataset with correct repo_id...

✅ Dataset loaded with repo_id: aryannzzz/metaworld-pick-place-v3-expert
✅ Episodes: 50
✅ Frames: 2,684

📤 Uploading Dataset to HuggingFace

   Repository: aryannzzz/metaworld-pick-place-v3-expert


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


✅ Dataset uploaded successfully!

🔗 View at: https://huggingface.co/datasets/aryannzzz/metaworld-pick-place-v3-expert



## 🎉 Dataset Generation Complete!

Your expert dataset is ready for ACT training. Next steps:

### Training Command
```bash
python lerobot/scripts/train.py \
    policy=act \
    env=metaworld \
    env.task=pick-place-v3 \
    dataset_repo_id=YOUR_USERNAME/metaworld-pick-place-v3-expert \
    training.offline_steps=100000 \
    wandb.enable=true
```

### Expected Performance
- **pick-place-v3**: 65-78% success after 100k steps
- **reach-v3**: 85-92% success
- **push-v3**: 75-85% success

## 🚀 Handle-Pull Task Dataset Generation

Generate dataset for **handle-pull-v3** task. Simply change `TASK_NAME` to generate datasets for other tasks.

**Valid task names:**
- `pick-place-v3` - Pick and place
- `push-v3` - Push  
- `reach-v3` - Reach
- `handle-pull-v3` - Pull a handle up
- `shelf-place-v3` - Place on shelf
- `pick-place-wall-v3` - Pick & place with wall

In [ ]:
# ==========================================
# GENERATE HANDLE-PULL-V3 DATASET
# ==========================================
# Change TASK_NAME to generate other tasks!

import os
import shutil
import numpy as np
from pathlib import Path

# --- CONFIGURATION ---
TASK_NAME = "handle-pull-v3"  # Change this for different tasks!
NUM_EPISODES = 50
MAX_EPISODE_STEPS = 500
HF_USERNAME = "aryannzzz"  # Your HuggingFace username

# HuggingFace repo ID for this dataset
DATASET_REPO_ID = f"{HF_USERNAME}/metaworld-{TASK_NAME}-expert"

print(f"\n{'='*70}")
print(f"🎬 Dataset Generation: {TASK_NAME}")
print(f"{'='*70}")
print(f"   Episodes: {NUM_EPISODES}")
print(f"   Max steps: {MAX_EPISODE_STEPS}")
print(f"   HF Repo: {DATASET_REPO_ID}")
print(f"{'='*70}\n")

In [ ]:
# ==========================================
# RECORD HANDLE-PULL DATASET
# ==========================================
# Reuses the same recording function from earlier cells

PULL_TASK = "handle-pull-v3"
PULL_EPISODES = 50

print(f"🚀 Starting dataset generation for {PULL_TASK}...")

# Record the dataset (uses the record_dataset function defined earlier)
pull_dataset_path = record_dataset(PULL_TASK, PULL_EPISODES)

print(f"\n✅ Dataset saved to: {pull_dataset_path}")

In [ ]:
# ==========================================
# UPLOAD HANDLE-PULL DATASET TO HUGGINGFACE
# ==========================================

from huggingface_hub import HfApi

# Configuration
PULL_TASK = "handle-pull-v3"
HF_REPO_ID = f"aryannzzz/metaworld-{PULL_TASK}-expert"
LOCAL_DATASET_PATH = ROOT_DIR / f"lerobot/{PULL_TASK}"

print(f"📤 Uploading dataset to HuggingFace Hub...")
print(f"   Repo: {HF_REPO_ID}")
print(f"   Local path: {LOCAL_DATASET_PATH}")

api = HfApi()

# Create repo if it doesn't exist
try:
    api.create_repo(repo_id=HF_REPO_ID, repo_type="dataset", exist_ok=True)
    print(f"   ✅ Repository ready")
except Exception as e:
    print(f"   ⚠️ Repo creation: {e}")

# Upload folder
api.upload_folder(
    folder_path=str(LOCAL_DATASET_PATH),
    repo_id=HF_REPO_ID,
    repo_type="dataset",
)

print(f"\n✅ Dataset uploaded successfully!")
print(f"   URL: https://huggingface.co/datasets/{HF_REPO_ID}")